# MultiHeadAttention 中的 `view` 与 `transpose` 维度变换

本 Notebook 演示在自注意力机制中，如何从输入序列通过单次线性投影生成 Queries/Keys/Values，并通过 `view` 与 `transpose` 完成多头隐式拆分。

### 多头注意力中的 `view` 隐式切分示例 (与 `MultiHeadAttention` 流程完全一致)

对应第 3 章完整计算流程：
1. 单样本 `inputs` 赋值并 `stack` 为 2 个批次的输入 `x`：形状为 `(2, 6, 3)`
2. 线性投影 `W_query = nn.Linear(d_in, d_out, bias=qkv_bias)`
3. 计算得到 `queries = W_query(x)`：形状为 `(2, 6, 2)`
4. 执行 `view(b, num_tokens, num_heads, head_dim)` 拆分多头：形状变为 `(2, 6, 2, 1)`
5. 执行 `transpose(1, 2)` 交换头维度与 token 维度：形状变为 `(2, 2, 6, 1)`

In [1]:
import torch
import torch.nn as nn

# 1. 初始输入 x (单个序列含 6 个 token，每个 token 嵌入维度 d_in = 3)
inputs = torch.tensor([[0.43, 0.15, 0.89],   # "Your"
                       [0.55, 0.87, 0.66],   # "journey"
                       [0.57, 0.85, 0.64],   # "starts"
                       [0.22, 0.58, 0.33],   # "with"
                       [0.77, 0.25, 0.10],   # "one"
                       [0.05, 0.80, 0.55]])  # "step"

# 转成 2 个批次: (b, num_tokens, d_in) = (2, 6, 3)
x = torch.stack((inputs, inputs), dim=0)
b, num_tokens, d_in = x.shape
print(f"输入 x 形状 (b={b}, num_tokens={num_tokens}, d_in={d_in}):", x.shape)

# 2. 超参数设置与线性层 W_query 赋值
d_out = 2        # 期望投影输出的维度
num_heads = 2    # 注意力头数
head_dim = d_out // num_heads  # 每个头的维度: 2 // 2 = 1
qkv_bias = False

torch.manual_seed(123)  # 固定随机种子以便复现
W_query = nn.Linear(d_in, d_out, bias=qkv_bias)

# 3. 线性变换得到 queries: (b, num_tokens, d_out) -> (2, 6, 2)
queries = W_query(x)
print(f"\n1. 经过 W_query 后的 queries 形状 (b, num_tokens, d_out):", queries.shape)
print("queries 具体内容:")
print(queries)

# 4. view 拆分出多头: (b, num_tokens, num_heads, head_dim) -> (2, 6, 2, 1)
queries_viewed = queries.view(b, num_tokens, num_heads, head_dim)
print(f"\n2. 仅执行 view 拆分后的形状 (b, num_tokens, num_heads, head_dim):", queries_viewed.shape)

# 5. transpose 交换维度: (b, num_heads, num_tokens, head_dim) -> (2, 2, 6, 1)
queries_transposed = queries_viewed.transpose(1, 2)
print(f"\n3. 接着执行 transpose(1, 2) 后的形状 (b, num_heads, num_tokens, head_dim):", queries_transposed.shape)
print("queries_transposed 具体内容:")
print(queries_transposed)


输入 x 形状 (b=2, num_tokens=6, d_in=3): torch.Size([2, 6, 3])

1. 经过 W_query 后的 queries 形状 (b, num_tokens, d_out): torch.Size([2, 6, 2])
queries 具体内容:
tensor([[[-0.3536,  0.3965],
         [-0.3021, -0.0289],
         [-0.3015, -0.0232],
         [-0.1353, -0.0978],
         [-0.2052,  0.0870],
         [-0.1542, -0.1499]],

        [[-0.3536,  0.3965],
         [-0.3021, -0.0289],
         [-0.3015, -0.0232],
         [-0.1353, -0.0978],
         [-0.2052,  0.0870],
         [-0.1542, -0.1499]]], grad_fn=<UnsafeViewBackward0>)

2. 仅执行 view 拆分后的形状 (b, num_tokens, num_heads, head_dim): torch.Size([2, 6, 2, 1])

3. 接着执行 transpose(1, 2) 后的形状 (b, num_heads, num_tokens, head_dim): torch.Size([2, 2, 6, 1])
queries_transposed 具体内容:
tensor([[[[-0.3536],
          [-0.3021],
          [-0.3015],
          [-0.1353],
          [-0.2052],
          [-0.1542]],

         [[ 0.3965],
          [-0.0289],
          [-0.0232],
          [-0.0978],
          [ 0.0870],
          [-0.1499]]],


        [[